# Лабораторная работа №1

## Компьютерная лексикография, терминологические базы (TBX) и Translation Memory (TMX) в Smartcat

**ФИО студента:** Шишкова Дарья Игоревна
**Группа:** ИНФА-221
**Вариант:** 14 (предметная область: "Строительство, архитектура и BIM-технологии")

---

**Цель:** научиться программно обрабатывать стандарты обмена терминологическими
данными (TBX) и памятью переводов (TMX), реализовать поиск нечётких совпадений
(Fuzzy Match) и подстановку терминов глоссария, экспортировать результат в
CAT-систему Smartcat.

**Порядок выполнения:**

1. Выполните ячейки Блока 1 (установка библиотек) и Блока 2 (данные).
2. Реализуйте четыре функции в Блоке 4 вместо маркеров `# TODO`.
3. Запустите Блок 5 — все `assert` должны пройти без ошибок.
4. Соберите файлы своего варианта, экспортируйте CSV, оформите отчёт.

**Оценивание (10 баллов):** парсинг TBX — 2, парсинг TMX — 2, Fuzzy Match — 3,
интеграция глоссария — 2, оформление — 1.

## Блок 1. Setup — установка библиотек

In [42]:
!pip install -q Levenshtein pandas lxml

In [43]:
from __future__ import annotations

import csv
import os
import re
import unicodedata
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import pandas as pd

# --- XML backend -----------------------------------------------------------
try:
    from lxml import etree as ET

    LXML_AVAILABLE = True
except ImportError:  # pragma: no cover - fallback для «голого» окружения
    import xml.etree.ElementTree as ET

    LXML_AVAILABLE = False

# --- Levenshtein backend ---------------------------------------------------
try:
    import Levenshtein as _lev

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна (C-реализация)."""
        return _lev.distance(a, b)

    LEV_BACKEND = "Levenshtein (C)"
except ImportError:  # pragma: no cover

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна (чистый Python, O(n*m) время, O(m) память)."""
        if a == b:
            return 0
        if not a:
            return len(b)
        if not b:
            return len(a)
        previous: List[int] = list(range(len(b) + 1))
        for i, char_a in enumerate(a, start=1):
            current: List[int] = [i]
            for j, char_b in enumerate(b, start=1):
                current.append(
                    min(
                        previous[j] + 1,          # удаление
                        current[j - 1] + 1,       # вставка
                        previous[j - 1] + (char_a != char_b),  # замена
                    )
                )
            previous = current
        return previous[-1]

    LEV_BACKEND = "pure-python fallback"

XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"
WORK_DIR = os.getcwd()


def indent_tree(element, level: int = 0, space: str = "  ") -> None:
    """Рекурсивно расставить отступы в XML-дереве (работает и с lxml, и с ET)."""
    pad = "\n" + space * level
    if len(element):
        if not (element.text or "").strip():
            element.text = pad + space
        for child in element:
            indent_tree(child, level + 1, space)
        if not (element.tail or "").strip():
            element.tail = pad
        if not (element[-1].tail or "").strip():
            element[-1].tail = pad
    elif level and not (element.tail or "").strip():
        element.tail = pad

pd.set_option("display.max_colwidth", 46)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

print(f"XML backend        : {'lxml' if LXML_AVAILABLE else 'xml.etree.ElementTree'}")
print(f"Levenshtein backend: {LEV_BACKEND}")
print(f"pandas             : {pd.__version__}")
print(f"Рабочий каталог    : {WORK_DIR}")

XML backend        : lxml
Levenshtein backend: Levenshtein (C)
pandas             : 2.2.3
Рабочий каталог    : /content


## Блок 2. Теория: структура стандартов

### 2.1. TBX (TermBase eXchange, ISO 30042)

Формат обмена терминологическими базами. Организован **по понятиям**
(concept-oriented): одна `termEntry` = одно понятие, внутри — по одной `langSet`
на каждый язык.

```xml
<martif type="TBX" xml:lang="en">
  <martifHeader>…</martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Предметная область</descrip>
      <langSet xml:lang="en">
        <tig>
          <term>flight controller</term>
          <termNote type="partOfSpeech">noun</termNote>
        </tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp>
          <term>полётный контроллер</term>
        </termGrp></ntig>
      </langSet>
    </termEntry>
  </body></text>
</martif>
```

| Тег | Назначение |
|-----|------------|
| `martif` | корень документа TBX-Core v2 |
| `termEntry` | терминологическая статья (одно понятие) |
| `langSet` | языковая секция, язык задаётся атрибутом `xml:lang` |
| `tig` / `ntig` | компактная / расширенная группа термина |
| `term` | сам термин |
| `termNote` | грамматическая или иная помета (`partOfSpeech`, `usage`) |
| `descrip` | описание: `definition`, `subjectField`, `context` |

### 2.2. TMX (Translation Memory eXchange, версия 1.4b)

Формат обмена памятью переводов. Организован **по сегментам**.

```xml
<tmx version="1.4">
  <header srclang="en" segtype="sentence" datatype="plaintext"
          creationtool="…" adminlang="en" o-tmf="TMX"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Aerospace</prop>
      <tuv xml:lang="en"><seg>The autopilot holds the altitude.</seg></tuv>
      <tuv xml:lang="ru"><seg>Автопилот выдерживает высоту.</seg></tuv>
    </tu>
  </body>
</tmx>
```

| Тег | Назначение |
|-----|------------|
| `header` | метаданные памяти: инструмент, язык оригинала, тип сегментации |
| `tu` | translation unit — единица перевода |
| `tuv` | translation unit variant — вариант на одном языке (`xml:lang`) |
| `seg` | текст сегмента |
| `prop` | произвольное свойство (`domain`, `client`, `project`) |

### 2.3. Типы совпадений в CAT-системах

| Совпадение | Диапазон | Действие переводчика |
|------------|----------|----------------------|
| ICE (101 %) | 100 % + совпал контекст | подстановка без правки |
| Exact | 100 % | проверить контекст |
| High Fuzzy | 95–99 % | косметическая правка |
| Fuzzy | 75–94 % | редактирование |
| No Match | < 75 % | перевод с нуля |

## Блок 3. Данные

Ячейка ниже создаёт **демонстрационные** файлы `demo.tbx` и `demo.tmx` — они
нужны только для автотестов из Блока 5. Файлы своего варианта
(`variant_NN.tbx` — не менее 15 терминов, `variant_NN.tmx` — не менее 10
предложений) вы создаёте самостоятельно: либо аналогичным кодом, либо выгрузкой
из Smartcat, либо вручную.

In [44]:
DEMO_TBX = """<?xml version="1.0" encoding="UTF-8"?>
<martif type="TBX" xml:lang="en">
  <martifHeader><fileDesc><titleStmt><title>Demo glossary</title></titleStmt>
  </fileDesc></martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en">
        <tig><term>flight controller</term>
        <termNote type="partOfSpeech">noun</termNote>
        <descrip type="definition">Onboard flight computer.</descrip></tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp><term>полётный контроллер</term></termGrp></ntig>
      </langSet>
    </termEntry>
    <termEntry id="tid-002">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>payload</term>
        <termNote type="partOfSpeech">noun</termNote></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>целевая нагрузка</term>
        </termGrp></ntig></langSet>
    </termEntry>
    <termEntry id="tid-003">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>telemetry link</term></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>канал телеметрии</term>
        </termGrp></ntig></langSet>
    </termEntry>
  </body></text>
</martif>
"""

DEMO_TMX = """<?xml version="1.0" encoding="UTF-8"?>
<tmx version="1.4">
  <header creationtool="demo" creationtoolversion="1.0" segtype="sentence"
          o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The flight controller stabilises the aircraft.</seg></tuv>
      <tuv xml:lang="ru"><seg>Полётный контроллер стабилизирует аппарат.</seg></tuv>
    </tu>
    <tu tuid="2">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The operator checks the telemetry link.</seg></tuv>
      <tuv xml:lang="ru"><seg>Оператор проверяет канал телеметрии.</seg></tuv>
    </tu>
    <tu tuid="3">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The payload is mounted under the fuselage.</seg></tuv>
      <tuv xml:lang="ru"><seg>Целевая нагрузка устанавливается под фюзеляжем.</seg></tuv>
    </tu>
  </body>
</tmx>
"""

with open("demo.tbx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TBX)
with open("demo.tmx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TMX)

print("Созданы файлы:", [f for f in os.listdir(".") if f.endswith((".tbx", ".tmx"))])

Созданы файлы: ['variant_14.tbx', 'variant_14.tmx', 'demo.tmx', 'demo.tbx']


In [45]:
TERMS: List[Dict[str, str]] = [
    {"en": "BIM module", "ru": "BIM-модуль", "pos": "noun",
     "def_en": "A digital model of an integrated structural unit that has the properties and characteristics of several structural elements designed in the information modeling environment."},
    {"en": "data centre", "ru": "центр обработки данных", "pos": "noun",
     "def_en": "A complex engineering structure, a high-tech site for maintaining information systems and telecommunications equipment, whose function is to process, store and transmit information, strategically important for a particular company, and manage its information system."},
    {"en": "prefabricated modular element", "ru": "модульный элемент максимальной готовности", "pos": "noun",
     "def_en": "A structural element made of different materials, which demonstrates the highest degree of readiness for installation."},
    {"en": "cloud-based", "ru": "облачные приложения", "pos": "adjective",
     "def_en": "Applications, services or resources made available on demand via the Internet from a cloud computing provider’s servers."},
    {"en": "family templates", "ru": "шаблоны семейства", "pos": "noun",
     "def_en": "Serve as building blocks for creating element families that contain the shared information needed to start a BIM modelling project."},
    {"en": "intellectual property", "ru": "интеллектуальная собственность", "pos": "noun",
     "def_en": "Creative work which can be treated as an asset or physical property."},
    {"en": "control points", "ru": "контрольные точки", "pos": "noun",
     "def_en": "A point on the ground or building that has pre-observed three dimensional co-ordinates."},
    {"en": "laser scanning", "ru": "лазерное сканирование", "pos": "noun",
     "def_en": "An active, fast and automatic acquisition technique using laser light for non-contact measurement of 3D coordinates of points on surfaces in a dense regular pattern."},
    {"en": "geophysics", "ru": "геофизика", "pos": "noun",
     "def_en": "The application of physics to study the earth and detect past human activity beneath the ground that plays a vital role in the discovery and understanding of archaeological sites."},
    {"en": "measured survey", "ru": "процесс изменений зданий и местности", "pos": "noun",
     "def_en": "The process of measurement for buildings and sites to provide an accurate, reduced scale representation showing the required structural elements and architectural features."},
    {"en": "metadata", "ru": "метаданные", "pos": "noun",
     "def_en": "Data that describes other data and facilitates the re-use and long-term preservation of 3D survey datasets."},
    {"en": "parametric modelling", "ru": "параметрическое моделирование", "pos": "noun",
     "def_en": "The creation of a digital model based on a series of pre-programmed rules so that changes in design will automatically update the assembly and its components."},
    {"en": "modelling tolerance", "ru": "допустимое значение моделирования", "pos": "noun",
     "def_en": "How accurately a model fits against the as-existing survey, usually a point cloud dataset."},
    {"en": "parametric objects", "ru": "параметрические объекты", "pos": "noun",
     "def_en": "Objects created using geometric definitions, associated data and rules that define their behaviour, how they interact with other objects or respond to changes in their parameters."},
    {"en": "photogrammetry", "ru": "фотограмметрия", "pos": "noun",
     "def_en": "The art, science and technology of determining size, shape and identification of objects by analysing terrestrial or aerial imagery."},
    {"en": "plug-ins", "ru": "подключаемые модули", "pos": "noun",
     "def_en": "A software component that adds a specific feature to an existing computer program."},
    {"en": "viewports", "ru": "видовые экраны", "pos": "noun",
     "def_en": "A users visible area of a page which can be single or multiple depending on the available viewing hardware and the process being undertaken."},
    {"en": "project north", "ru": "вид сверху", "pos": "noun",
     "def_en": "The top of the drawing area within BIM authoring software."},
    {"en": "point cloud", "ru": "облако точек", "pos": "noun",
     "def_en": "A collection of XYZ coordinates in a common co-ordinate system, that may also include additional information such as an intensity or RGB value, that portrays to the viewer an understanding of the spatial distribution of the surface of a subject."},
    {"en": "carbon performance", "ru": "углеродные показатели", "pos": "noun",
     "def_en": "A measure of the carbon emissions from the operation, use and maintenance of an asset and the emissions associated with the users of the asset."},
    {"en": "desk-based assessment", "ru": "кабинетная оценка", "pos": "noun",
     "def_en": " The summary, collation and/or synthesis of existing research, assessments and management of legacy information."},
    {"en": "federated discipline-specific models", "ru": "интегрированные отраслевые модели", "pos": "noun",
     "def_en": "An assembly of the discipline-specific models that contain 3D geometric and non-graphical data and associated documentation."},
    {"en": "intangible characteristics", "ru": "нематериальные характеристики", "pos": "noun",
     "def_en": "Identifiable non-monetary assets, such as heritage values and significance, that cannot be seen, touched or physically measured and are created through time and effort."},
    {"en": "meshing", "ru": "создание сетки", "pos": "noun",
     "def_en": "The process of generating a polygonal or polyhedral mesh through the interconnection of three dimensional vertices."},
    {"en": "metric survey", "ru": "метрический обзор", "pos": "noun",
     "def_en": "The use of precise and repeatable measurement methods to capture spatial information for reproduction at scale."},
    {"en": "native BIM", "ru": "встроенное BIM-моделирование", "pos": "noun",
     "def_en": "The original component creation process used for representing the geometry of existing fabric."},
    {"en": "operation and maintenance (O&M)", "ru": "эксплуатация и техническое обслуживание", "pos": "noun",
     "def_en": "The daily activities and services needed to ensure an asset performs to its intended function."},
    {"en": "operational stage", "ru": "операционный этап", "pos": "noun",
     "def_en": "The final stage in an assets development when the BIM data is used to manage all the information related to the operation and maintenance of the asset."},
    {"en": "professional liability", "ru": "профессиональная ответственность", "pos": "noun",
     "def_en": "Legal obligations arising out of a professional’s errors, acts of negligence or omissions during the course of their work."},
    {"en": "anthropometry", "ru": "антропометрия", "pos": "noun",
     "def_en": "The measurement and study of the size and proportions of the human body."},
    {"en": "atrium", "ru": "атрий", "pos": "noun",
     "def_en": "Originally, the main or central inner hall of an ancient Roman house, open to the sky at the center and usually having a pool for the collection of rainwater. Later, the forecourt of an early Christian church, flanked or surrounded by porticoes. Now, an open, skylit court around which a house or building is built."},
    {"en": "balcony", "ru": "балкон", "pos": "noun",
     "def_en": "An elevated platform projecting from the wall of a building and enclosed by a railing or parape."},
    {"en": "facade", "ru": "фасад", "pos": "noun",
     "def_en": "The front of a building or any of its sides facing a public way or space, esp. one distinguished by its architectural treatment."},
    {"en": "semiotics", "ru": "семиотика", "pos": "noun",
     "def_en": "The study of signs and symbols as elements of communicative behavior."},
    {"en": "pyramid", "ru": "пирамида", "pos": "noun",
     "def_en": "A massive masonry structure having a rectangular base and four smooth, steeply sloping sides facing the cardinal points and meeting at an apex, used in ancient Egypt as a tomb to contain the burial chamber and the mummy of the pharaoh."},
    {"en": "harmony", "ru": "гармония", "pos": "noun",
     "def_en": "The orderly, pleasing, or congruent arrangement of the elements or parts in an artistic whole."},
    {"en": "golden section", "ru": "золотое сечение", "pos": "noun",
     "def_en": "A proportion between the two dimensions of a plane figure or the two divisions of a line, in which the ratio of the smaller to the larger is the same as the ratio of the larger to the whole: a ratio of approximately 0.618 to 1.000."},
    {"en": "fresco", "ru": "фреска", "pos": "noun",
     "def_en": "The art or technique of painting on a freshly spread, moist plaster surface with pigments ground up in water or a limewater mixtur."},
    {"en": "frieze", "ru": "фриз", "pos": "noun",
     "def_en": "The horizontal part of a classical entablature between the cornice and architrave, often decorated with sculpture in low relief. Also, a decorative band, as one along the top of an interior wall, immediately below the cornice, or a sculptured one in a stringcourse on an outside wall."},
    {"en": "gallery", "ru": "галерея", "pos": "noun",
     "def_en": "A long, relatively narrow room or hall, esp. one for public use and having architectural importance through its scale or decorative treatment. Also, a roofed promenade, esp. one extending inside or outside along the exterior wall of a building."},
    {"en": "rhythm", "ru": "ритм", "pos": "noun",
     "def_en": "Movement characterized by a patterned repetition or alternation of formal elements or motifs in the same or a modified form."},
    {"en": "roof", "ru": "крыша", "pos": "noun",
     "def_en": "The external upper covering of a building, including the frame for supporting the roofing."},
    {"en": "terrace", "ru": "терраса", "pos": "noun",
     "def_en": "A raised level with a vertical or sloping front or sides faced with masonry, turf, or the like, esp. one of a series of levels rising above one another."},
    {"en": "vestibule", "ru": "вестибюль", "pos": "noun",
     "def_en": "A small entrance hall between the outer door and the interior of a house or building."},
    {"en": "wall", "ru": "стена", "pos": "noun",
     "def_en": "Any of various upright constructions presenting a continuous surface and serving to enclose, divide, or protect an area."},
    {"en": "villa", "ru": "вилла", "pos": "noun",
     "def_en": "A country residence or estate."},
    {"en": "topography", "ru": "топография", "pos": "noun",
     "def_en": "The physical configuration and features of a site, area, or region."},
    {"en": "transformation", "ru": "трансформация", "pos": "noun",
     "def_en": "The process of changing in form or structure through a series of discrete permutations and manipulations in response to a specific context or set of conditions without a loss of identity or concept."},
    {"en": "volume", "ru": "объем", "pos": "noun",
     "def_en": "The size or extent of a three-dimensional object or region of space, measured in cubic units."},
    {"en": "space", "ru": "пространство", "pos": "noun",
     "def_en": "The three-dimensional field in which objects and events occur and have relative position and direction, esp. a portion of that field set apart in a given instance or for a particular purpose."},
    {"en": "anomaly", "ru": "аномалия", "pos": "noun",
     "def_en": "A deviation from the normal or expected form, order, or arrangement."},
    {"en": "axis", "ru": "ось", "pos": "noun",
     "def_en": "A central line that bisects a two-dimensional body or figure or about which a three-dimensional body or figure is symmetrical. Also, a straight line to which elements in a composition are referred for measurement or symmetry."},
    {"en": "concrete", "ru": "бетон", "pos": "noun",
     "def_en": "An artificial, stonelike building material made by mixing cement and various mineral aggregates with sufficient water to cause the cement to set and bind the entire mass."},
    {"en": "ergonomics", "ru": "эргономика", "pos": "noun",
     "def_en": "An applied science concerned with the characteristics of people that need to be considered in the design of devices and systems in order that people and things will interact effectively and safely."},
    {"en": "capital", "ru": "капитель", "pos": "noun",
     "def_en": "The distinctively treated upper end of a column, pillar, or pier, crowning the shaft and taking the weight of the entablature or architrave."},
    {"en": "figure", "ru": "фигура", "pos": "noun",
     "def_en": "A shape or form, as determined by outlines or exterior surfaces. Also, a combination of geometric elements disposed in a particular form or shape."},
    {"en": "colonnade", "ru": "колоннада", "pos": "noun",
     "def_en": " A series of regularly spaced columns supporting an entablature and usually one side of a roof structure."},
    {"en": "column", "ru": "колонна", "pos": "noun",
     "def_en": "A rigid, relatively slender structural member designed primarily to support compressive loads applied at the member ends. In classical architecture, a cylindrical support consisting of a capital, shaft, and usually a base, either monolithic or built up of drums the full diameter of the shaft."},
    {"en": "Building Information Modelling (BIM)", "ru": "BIM (Информационное моделирование зданий)", "pos": "noun",
     "def_en": "A collaborative process for the production and management of structured electronic information and illustrating, in digital terms, all the elements that compose a building."},
    {"en": "Asset Information Requirements (AIR)", "ru": "требования к информации об активах", "pos": "noun",
     "def_en": "Define the information required at project handover stage to enable the safe and effective operation of the asset."},
    {"en": "Project Information Model (PIM)", "ru": "информационная модель проекта", "pos": "noun",
     "def_en": "developed during the design and construction phase of a project which often forms the basis of the asset information model."},
    {"en": "church", "ru": "храм (церковь)", "pos": "noun",
     "def_en": "A building for public Christian worship."},
    {"en": "computer modeling", "ru": "компьютерное моделирование", "pos": "noun",
     "def_en": "The use of computer technology and mathematical algorithms to create abstract models of systems and processes to simulate their behavior."},
    {"en": "cantilever", "ru": "консоль", "pos": "noun",
     "def_en": "A beam or other rigid structural member extending beyond a fulcrum and supported by a balancing member or a downward force behind the fulcrum."},
    {"en": "module", "ru": "модуль", "pos": "noun",
     "def_en": "A unit of measurement used for standardizing the dimensions of building materials or regulating the proportions of an architectural composition."},
    {"en": "contrast", "ru": "контраст", "pos": "noun",
     "def_en": "Opposition or juxtaposition of dissimilar elements in a work of art to intensify each element’s properties and produce a more dynamic expressiveness."},
    {"en": "podium", "ru": "подиум", "pos": "noun",
     "def_en": "A solid mass of masonry visible above ground level and serving as the foundation of a building, esp. the platform forming the floor and substructure of a classical temple."},
    {"en": "unity", "ru": "единство", "pos": "noun",
     "def_en": "The state or quality of being combined into one, as the ordering of elements in an artistic work that constitutes a harmonious whole or promotes a singleness of effect."},
    {"en": "torii", "ru": "тории", "pos": "noun",
     "def_en": "A monumental, freestanding gateway on the approach to a Shinto shrine, consisting of two pillars connected at the top by a horizontal crosspiece and a lintel above it, usually curving upward."},
    {"en": "tokonoma", "ru": "токонома", "pos": "noun",
     "def_en": "Picture recess: a shallow, slightly raised alcove for the display of a flower arrangement or a kakemono, a vertical hanging scroll containing either text or a painting."},
    {"en": "threshold", "ru": "порог", "pos": "noun",
     "def_en": "A place or point of entering or beginning."},
    {"en": "tetrastyle", "ru": "тетрастиль", "pos": "noun",
     "def_en": "Having four columns on one or each front."},
    {"en": "symmetry", "ru": "симметрия", "pos": "noun",
     "def_en": "The exact correspondence in size, form, and arrangement of parts on opposite sides of a dividing line or plane, or about a center or axis. Also, regularity of form or arrangement in terms of like, reciprocal, or corresponding parts."},
    {"en": "story", "ru": "этаж", "pos": "noun",
     "def_en": "A complete horizontal division of a building, having a continuous or nearly continuous floor and comprising the space between two adjacent levels."},
    {"en": "stele", "ru": "стела", "pos": "noun",
     "def_en": "An upright stone slab or pillar with a carved or inscribed surface, used as a monument or marker, or as a commemorative tablet in the face of a building."},
    {"en": "spire", "ru": "шпиль", "pos": "noun",
     "def_en": "A tall, acutely tapering pyramidal structure surmounting a steeple or tower."},
    {"en": "sill", "ru": "подоконник", "pos": "noun",
     "def_en": "The lowest horizontal member of a frame structure, resting on and anchored to a foundation wall. Also, the horizontal member beneath a door or window opening."},
    {"en": "shrine", "ru": "святилище", "pos": "noun",
     "def_en": "A building or other shelter, often of a stately or sumptuous character, enclosing the remains or relics of a saint or other holy person and forming an object of religious veneration and pilgrimage."},
    {"en": "proportion", "ru": "пропорция", "pos": "noun",
     "def_en": "The comparative, proper, or harmonious relation of one part to another or to the whole with respect to magnitude, quantity, or degree. Also, the equality between two ratios in which the first of the four terms divided by the second equals the third divided by the fourth."},
    {"en": "rampart", "ru": "вал (крепостной)", "pos": "noun",
     "def_en": "A broad embankment of earth raised as a fortification around a place and usually surmounted by a parapet."},
    {"en": "promenade", "ru": "променад", "pos": "noun",
     "def_en": "An area used for a stroll or walk, esp. in a public place, as for pleasure or display."},
    {"en": "portico", "ru": "портик", "pos": "noun",
     "def_en": "A porch or walkway having a roof supported by columns, often leading to the entrance of a building."},
    {"en": "peristyle", "ru": "перистиль", "pos": "noun",
     "def_en": "A colonnade surrounding a building or a courtyard. Also, the courtyard so enclosed."},
    {"en": "pavilion", "ru": "павильон", "pos": "noun",
     "def_en": "A light, usually open building used for shelter, concerts, or exhibits, as in a park or fair. Also, a central or flanking projecting subdivision of a facade, usually accented by more elaborate decoration or greater height and distinction of skyline."},
    {"en": "palazzo", "ru": "палаццо", "pos": "noun",
     "def_en": "A large, imposing public building or private residence, esp. in Italy."},
    {"en": "pagoda", "ru": "пагода", "pos": "noun",
     "def_en": "A Buddhist temple in the form of a square or polygonal tower with roofs projecting from each of its many stories, erected as a memorial or to hold relics."},
    {"en": "obelisk", "ru": "обелиск", "pos": "noun",
     "def_en": "A tall, four-sided shaft of stone that tapers as it rises to a pyramidal point, originating in ancient Egypt as a sacred symbol of the sun-god Ra and usually standing in pairs astride temple entrances."},
    {"en": "nuraghe", "ru": "нураги", "pos": "noun",
     "def_en": "Any of the large, round or triangular stone towers found in Sardinia and dating from the second millennium B.C. to the Roman conquest."},
    {"en": "niche", "ru": "ниша", "pos": "noun",
     "def_en": "An ornamental recess in a wall, often semicircular in plan and surmounted by a half-dome, as for a statue or other decorative object."},
    {"en": "nave", "ru": "корабль (нэф)", "pos": "noun",
     "def_en": "The principal or central part of a church, extending from the narthex to the choir or chancel and usually flanked by aisles."},
    {"en": "mosque", "ru": "мечеть", "pos": "noun",
     "def_en": "A Muslim building or place of public worship."},
    {"en": "minaret", "ru": "минарет", "pos": "noun",
     "def_en": "A lofty, slender tower attached to a mosque, having stairs leading up to one or more projecting balconies from which the muezzin calls the Muslim people to prayer."},
    {"en": "mezzanine", "ru": "мезонин", "pos": "noun",
     "def_en": "A low or partial story between two main stories of a building, esp. one that projects as a balcony and forms a composition with the story beneath it."},
    {"en": "megalith", "ru": "мегалит", "pos": "noun",
     "def_en": "A very large stone used as found or roughly dressed, esp. in ancient construction work."},
    {"en": "mausoleum", "ru": "мавзолей", "pos": "noun",
     "def_en": "A large and stately tomb, esp. that in the form of a building for housing the tombs of many individuals, often of a single family."},
    {"en": "mass", "ru": "масса", "pos": "noun",
     "def_en": "The physical volume or bulk of a solid body."},
    {"en": "joist", "ru": "балка", "pos": "noun",
     "def_en": "Any of a series of small, parallel beams for supporting floors, ceilings, or flat roofs."},
    {"en": "hierarchy", "ru": "иерархия", "pos": "noun",
     "def_en": "A system of elements ranked, classified, and organized one above another, according to importance or significance."},
    {"en": "hall", "ru": "холл", "pos": "noun",
     "def_en": "The large entrance room of a house or building, as a vestibule or lobby. Also, a large room or building for public gatherings or entertainment."},
    {"en": "background", "ru": "фон", "pos": "noun",
     "def_en": "The part of an image represented as being at the maximum distance from the frontal plane."}
]

SUBJECT_FIELD = "Строительство, архитектура и BIM-технологии"
TBX_PATH = "variant_14.tbx"


def build_tbx(terms: Sequence[Dict[str, str]], path: str,
              subject_field: str = SUBJECT_FIELD) -> str:
    """Сгенерировать валидный TBX-файл (TBX-Core v2, martif).

    Args:
        terms: последовательность словарей с ключами ``en``, ``ru``, ``pos``, ``def_en``.
        path: путь к создаваемому файлу.
        subject_field: предметная область, записывается в ``descrip[subjectField]``.

    Returns:
        Путь к созданному файлу.
    """
    martif = ET.Element("martif", {"type": "TBX", XML_LANG: "en"})

    header = ET.SubElement(martif, "martifHeader")
    file_desc = ET.SubElement(header, "fileDesc")
    title_stmt = ET.SubElement(file_desc, "titleStmt")
    ET.SubElement(title_stmt, "title").text = f"Glossary EN-RU: {subject_field}"
    source_desc = ET.SubElement(file_desc, "sourceDesc")
    ET.SubElement(source_desc, "p").text = "Lab 01, variant 14, generated automatically"
    encoding_desc = ET.SubElement(header, "encodingDesc")
    ET.SubElement(encoding_desc, "p", {"type": "XCSURI"}).text = "TBXXCSV02.xcs"

    body = ET.SubElement(ET.SubElement(martif, "text"), "body")

    for index, item in enumerate(terms, start=1):
        entry = ET.SubElement(body, "termEntry", {"id": f"tid-{index:03d}"})
        ET.SubElement(entry, "descrip", {"type": "subjectField"}).text = subject_field

        # --- английская секция: краткая форма <tig> ---
        lang_en = ET.SubElement(entry, "langSet", {XML_LANG: "en"})
        tig = ET.SubElement(lang_en, "tig")
        ET.SubElement(tig, "term").text = item["en"]
        ET.SubElement(tig, "termNote", {"type": "partOfSpeech"}).text = item["pos"]
        ET.SubElement(tig, "descrip", {"type": "definition"}).text = item["def_en"]

        # --- русская секция: расширенная форма <ntig><termGrp> ---
        lang_ru = ET.SubElement(entry, "langSet", {XML_LANG: "ru"})
        term_grp = ET.SubElement(ET.SubElement(lang_ru, "ntig"), "termGrp")
        ET.SubElement(term_grp, "term").text = item["ru"]
        ET.SubElement(term_grp, "termNote", {"type": "partOfSpeech"}).text = item["pos"]

    indent_tree(martif)
    xml_body = ET.tostring(martif, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write('<!DOCTYPE martif SYSTEM "TBXcoreStructV02.dtd">\n')
        handler.write(xml_body)
    return path


build_tbx(TERMS, TBX_PATH)
print(f"Создан {TBX_PATH}: {os.path.getsize(TBX_PATH)} байт, терминов: {len(TERMS)}\n")
print("--- первые 26 строк файла ---")
with open(TBX_PATH, encoding="utf-8") as handler:
    for line in list(handler)[:26]:
        print(line.rstrip())

Создан variant_14.tbx: 79232 байт, терминов: 100

--- первые 26 строк файла ---
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE martif SYSTEM "TBXcoreStructV02.dtd">
<martif type="TBX" xml:lang="en">
  <martifHeader>
    <fileDesc>
      <titleStmt>
        <title>Glossary EN-RU: Строительство, архитектура и BIM-технологии</title>
      </titleStmt>
      <sourceDesc>
        <p>Lab 01, variant 14, generated automatically</p>
      </sourceDesc>
    </fileDesc>
    <encodingDesc>
      <p type="XCSURI">TBXXCSV02.xcs</p>
    </encodingDesc>
  </martifHeader>
  <text>
    <body>
      <termEntry id="tid-001">
        <descrip type="subjectField">Строительство, архитектура и BIM-технологии</descrip>
        <langSet xml:lang="en">
          <tig>
            <term>BIM module</term>
            <termNote type="partOfSpeech">noun</termNote>
            <descrip type="definition">A digital model of an integrated structural unit that has the properties and characteristics of several structur

In [46]:
PARALLEL_CORPUS: List[Tuple[str, str]] = [
    ("The main characteristic of a BIM module is the representation of all components within a single structural unit, i.e. as a block having any geometric shape.",
     "Главная характеристика BIM-модуля — представление всех составляющих в одной структурной единице, т.е. в виде блока в любом геометрическом виде."),
    ("The data centre infrastructure includes several engineering systems with a high level of reliability.",
     "В инфраструктуру центра обработки данных входит несколько инженерных систем высокого уровня надежности."),
    ("The purpose of this work is to determine the effectiveness of design based on prefabricated modular elements (PME) using information modeling tools applied to a construction facility.",
     "Цель данной работы — определение эффективности проектирования на основе модульных элементов (МЭ) максимальной готовности с помощью средств информационного моделирования на примере конкретного объекта строительства."),
    ("Cloud-based applications and services are becoming increasingly more popular.",
     "Облачные приложения и сервисы становятся все более популярными."),
    ("High-resolution scans were taken that were meshed using third-party software then built into the model using standard Revit family templates, to maintain a reasonable model size and avoid the need for importing non-Revit formats such as DWG files directly into the model space.",
     "Было проведено сканирование высокого разрешения, которое было интегрировано с помощью стороннего программного обеспечения, встроенного в модель с помощью стандартных шаблонов семейства Revit, чтобы поддерживать разумный размер модели и избежать необходимости импортировать иные форматы, кроме Revit (например, файлы в формате DWG) непосредственно в пространство модели."),
    ("Existing information may be in paper or electronic format, on-site or dispersed in several off-site locations, and potentially under intellectual property or security restrictions.",
     "Существующая информация может быть предоставлена в бумажном или электронном формате, на месте или рассредоточена в нескольких местах, и потенциально представлять собой объект интеллектуальной собственности или являться конфиденциальной информацией."),
    ("A network of control points was installed and coordinated to the UK Ordnance Survey (OS) national grid using traverse methods and global positioning system (GPS) virtual reference stations (VRS).",
     "Была установлена и скоординирована сеть контрольных точек для национальной сети картографической съемки Великобритании (OS) с использованием методов траверса и виртуальных опорных станций глобальной системы позиционирования (GPS) (VRS)."),
    ("Laser scanning, photogrammetry (groundbased or mounted on a drone), lidar, closer range scanning, mobile mapping or a combination of methods can be used to produce 3D datasets of the historic asset.",
     "Лазерное сканирование, фотограмметрия (наземная или установленная на беспилотнике), работа с лидаром (LIDAR англ. Light Identification Detection and Ranging «обнаружение, идентификация и определение дальности с помощью света»), сканирование на близком расстоянии, мобильное картирование или комбинация методов может использоваться для создания 3D наборов данных исторического здания."),
    ("Integration of heterogeneous datasets, such as historic information, legacy data, photographs and drawings, geospatial datasets, geophysics and remotely sensed data and imagery.",
     "Интеграция разнородных источников данных, таких как исторические источники, данные в устаревшем формате, фотоснимки и чертежи, геопространственные данные, геофизические данные и данные дистанционного зондирования."),
    ("Andrews et al (2015) provides guidance and template specifications for a range of survey methods, such as image-based, measured survey and laser scanning.",
     "В издании Andrews et al (2015) предоставлено техническое руководство и шаблоны спецификаций для ряда методов исследований, основанных на изображении, обследовании с замерами здания и лазерном сканировании."),
    ("Naming conventions are crucial for information management processes, especially in the absence of a data management system able to support metadata.",
     "Соглашения о названиях имеют решающее значение для процессов управления информацией, особенно в ситуации отсутствия системы управления данными, способной поддерживать метаданные."),
    ("In parametric modelling, changes in design will automatically update the assembly and its components so that rules and object definitions are always valid.",
     "В параметрическом моделировании изменения в конструкции автоматически влияют на модель и ее компоненты, поэтому правила и определения объектов всегда являются верными."),
    ("Modelling tolerance refers to how accurately a model fits against the as-existing survey (usually a point cloud dataset).",
     "Допустимое значение моделирования относится к тому, насколько точно модель соответствует существующему обследованию (обычно набору данных облака точек)."),
    ("BIM software restrictions also make the use of native parametric objects for irregular geometries difficult or impossible to achieve without limiting model functionality.",
     "Ограничения программного обеспечения BIM также затрудняют или делают невозможными использование родных параметрических предметов неправильной геометрической формы без ограничения функциональности модели."),
    ("Articles relevant to Historic BIM can be found in the International Society for Photogrammetry and Remote Sensing (ISPRS) archive and the International Committee for Documentation of Cultural Heritage (CIPA) conference archives.",
     "Статьи, имеющие отношение к BIM-моделированию исторических объектов, можно найти на сайте Международного общества фотограмметрии и дистанционного зондирования (ISPRS) и в Архиве конференции Международного Комитета по документации культурного наследия (CIPA)."),
    ("A number of plug-ins and additional tools are available to cover a range of applications, such as automatic geometry extraction (for scanto-BIM workflows), clash detection, project management, visualisation, interfacing with other systems, and others.",
     "Множество плагинов и дополнительных инструментов способны обеспечить такие области применения, как автоматическое извлечение геометрии (для потоков операций сканирования для BIM), обнаружение коллизий, управление проектом, визуализация, взаимодействие с другими системами и др."),
    ("Within the issued file, sheets were set up for each floorplan, including the roof level, along with elevations and typical sections and rendered viewports throughout the structure.",
     "В выданном файле для каждого плана этажа были настроены отдельные листы, включая уровень крыши, а также разверстки и визуализированные окна конструирования по всему зданию."),
    ("The final model was released to the client as a Revit model set up with both a local grid on a prearranged project north, and the UK Ordnance Survey (OS) national grid.",
     "Окончательная модель была выпущена для клиента в виде модели Revit, настроенной как с наличием локальной сетки на заранее подготовленный проект, так и с национальная сетью картографической службы Великобритании."),
    ("These typically take the form of point clouds, which are now supported by most BIM software.",
     "Обычно они преобразуются в облака точек, которые теперь поддерживаются большинством компьютерных программ для BIM."),
    ("Construction projects in the heritage sector (conservation refurbishment, adaptive reuse, extension and repair) could similarly benefit from the adoption of BIM and collaborative working processes, with increased efficiency, reduced costs, better planning and improved carbon performance for historic buildings and sites.",
     "Строительные проекты в сфере сохранения наследия (консервация и восстановление, адаптивное повторное использование, расширение и ремонт) могли бы аналогичным образом выиграть при условии использования BIM и совместных рабочих процессов, чтобы привело бы к повышению эффективности, снижению затрат, улучшению планирования и оптимизации выработки углерода для исторических зданий и других объектов."),
    ("Further investigations, qualitative surveys, desk-based assessments and site inspections can be required depending on the scope of the project and client requirements.",
     "В зависимости от объема проекта и требований клиента могут потребоваться дальнейшие обследования объекта, качественные и кабинетные исследования, а также инспекции на местах."),
    ("BIM Level 2 refers to a collaborative process of producing federated disciplinespecific models, consisting of 3D geometric and non-graphical data and associated documentation.",
     "BIM 2-го уровня подразумевает совместный процесс создания интегрированных специализированных моделей, состоящих из данных в форме 3D, геометрических и неграфических данных и соответствующей документации."),
    ("Intangible characteristics, such as heritage values and significance, can be integrated into the 3D model in a structured and consistent way, which allows easy information extraction and the production of deliverables.",
     "Такие нематериальные характеристики объекта, как его историческая значимость и ценность в аспекте наследия, могут быть последовательно интегрированы в 3D-модель, что позволяет просто извлекать информацию и проектную документацию."),
    ("Meshing techniques were employed by the modelling team to enable the creation of the more ‘organic’ geometry presented by these objects.",
     "Команда моделирования использована метод создания сетки для репрезентации более «органической» геометрии этих объектов."),
    ("Historic BIM should be based on accurate as-existing metric survey datasets (preferably 3D), which document the position, size and dimensions of all visible surfaces, components and context of the historic asset, referenced to a local or national coordinate system.",
     "Информационное моделирование исторического объекта должно быть основано на точных и реальных наборах данных метрических измерений (предпочтительно в форме 3D-модели), которые фиксируют расположение, размер и объемы всех видимых поверхностей, строительных элементов и особенностей, характерных для исторического объекта, на которые ссылаются местная или национальная система координат."),
    ("This can be a relatively quick way to add complex geometry to a model, without the need to recreate the object as native BIM geometry.",
     "Он может позволить относительно быстро добавить сложные геометрические элементы к модели без необходимости воссоздавать объект в качестве родной геометрии BIM."),
    ("These can include archival data, product specifications, operation and maintenance (O&M) manuals, reports, condition surveys, audio and video recordings (eg documenting visitor experience), inspection logs or any other type of digital file.",
     "К ним можно отнести архивные данные, технические характеристики изделий, руководства по эксплуатации и техническому обслуживанию, отчеты, обследования состояния объекта, аудиои видеозаписи (например, задокументированный опыт посетителя), журналы проверки технического состояния или любой другой тип цифрового файла."),
    ("BIM provides a methodology for better information management on design and construction projects, with eventual handover of the full information required for the operational stage.",
     "BIM обеспечивает методологию для лучшего управления информацией по проектированию и строительным проектам, с возможной передачей полной информации, необходимой для строительного этапа."),
    ("The subject of IPR and professional liability on BIM projects is complex.",
     "Тема интеллектуальной собственности и профессиональной ответственности в проектах BIM является сложной."),
    ("Anthropometry refers to the measurement of the size and proportions of the human body.",
     "Антропометрия имеет дело с размерами и пропорциями человеческого тела."),
    ("They are manifested here in the atrium spaces of houses, the arcaded cortile of an Italian palazzo, the enclosure of a Greek shrine, the courtyard of a Finnish town hall, and the cloister of a monastery.",
     "Здесь они представлены атриумным пространством домов, обнесенным аркадами итальянского палаццо, внутренним пространством греческого святилища, внутренним домом финской городской ратуши, клуатром монастыря."),
    ("A stairway leading up to a second-story balcony or loft can give us an idea of the vertical dimension of a room as well as suggest a human presence.",
     "Лестница, ведущая на балкон второго этажа, дает представление о высоте помещения и одновременно намекает на человеческое присутствие."),
    ("The Renaissance facade by Alberti presents a public face to a square.",
     "Парадный фасад эпохи Ренессанса по проекту Альберти выходит на городскую площадь."),
    ("Although the study of connotative meanings, of semiotics and symbology in architecture, is beyond the scope of this book.",
     "Хотя рассмотрение смысловых значений, семиотики и символики в архитектуре выходит за рамки этой книги, здесь необходимо сказать, что архитектура, сочетая в единую сущность форму и пространство, не только оформляет функциональные потребности, но и передает смысл."),
    ("The pyramid has properties similar to those of the cone.",
     "По своим качествам пирамида более всего сходна с конусом."),
    ("The intent of all theories of proportions is to create a sense of order and harmony among the elements in a visual construction.",
     "Цель всех пропорциональных теорий - создать впечатление упорядоченности и гармонии всех зримых элементов постройки."),
    ("The Golden Section has some remarkable algebraic and geometric properties that account for its existence in architecture as well as in the structures of many living organisms.",
     "У золотого сечения есть несколько примечательных алгебраических и геометрических особенностей, которые отражаются и в архитектуре, и в строении многих живых организмов."),
    ("It can serve as a repository for frescoes and other means of artistic expression or be treated simply as a passive or receding surface.",
     "Потолок может быть выразительной художественной зоной, размещающей фрески или другое художественное оформление."),
    ("And for each room size, a different ceiling height is established according to the following: height of the ceiling (shaku), measured from the top of the frieze board = number of mats x 0.3.",
     "Также для каждого помещения устанавливается своя высота потолка, исходя из следующего правила: высота потолка (сяку) от верхней поверхности балки фриза равняется числу матов, умноженному на 0,3."),
    ("The enclosure may consist of arcades or gallery spaces that promote the inclusion of surrounding buildings into their domain and activate the space they define.",
     "Оно может быть обнесено аркадами или галереями, пространство которых активизирует связь между окружающими сооружениями и ограждаемым пространством."),
    ("A row of columns can also engage a wall and become a pilastrade that supports the wall, articulates its surface, and tempers the scale, rhythm, and proportioning of its bays.",
     "Ряд колонн может также примыкать к стене, превращаясь в пилястры, которые способствуют конструктивному укреплению стены и художественной разработке ее поверхности, задавая масштаб, ритм и пропорции ее членений."),
    ("As a design element, the roof plane is significant because of the impact it can have on the form and silhouette of a building within its setting.",
     "Как элемент дизайна плоскость крыши имеет огромное значение в силу того влияния, которое она способна оказывать на форму и силуэт сооружения."),
    ("A garden terrace distributes sunlight to the spaces gathered around it.",
     "Садовая терраса равномерно распределяет солнечный свет в окружающие ее пространства."),
    ("During the Renaissance, Andrea Palladio incorporated the tetrastyle theme in the vestibules and halls of a number of villas and palazzi.",
     "В эпоху Ренессанса Андреа Палладио в целом ряде своих вилл и дворцов ввел тему тетрастиля в разработку вестибюлей и залов."),
    ("A row of columns is indeed nothing but a wall, open and discontinued in several places.",
     "В сущности, ряд колонн - не что иное, как стена, открытая и в нескольких местах разорванная»."),
    ("Within the Villa Foscari you are aware of the thickness of the walls that separate the rooms, each of which has been given definitive and precise form.",
     "В вилле Фоскари ощущается тотцина стен, разделяющих помещения, каждому из которых придана четкая точная форма."),
    ("To respond to contrasting geometries of the topography, vegetation, boundaries, or existing structures of a site.",
     "Чтобы приспособиться к контрастным геометрическим очертаниям топографии местности, растительности, границам и структуре участка."),
    ("The diagrams illustrate the transformation of a row of round columns, initially supporting a portion of a wall, then evolving into square piers which are an integral part of the wall plane, and finally becoming pilasters—remnants of the original columns occurring as a relief along the surface of the wall.",
     "На схемах показана трансформация ряда круглых колонн, изначально несущих часть стены, которые потом превращаются в квадратные в плане пилоны, сами становясь частью стенной плоскости, и, наконец, в пилястры - остатки прежних колонн, - существующие на стенной плоскости уже только в виде рельефа."),
    ("Four exterior wall planes define a rectangular volume that contains the program elements and spaces.",
     "Четыре наружных стенных панели образуют прямоугольный объем, в котором заключаются программные элементы и пространства."),
    ("In the movement from plane to spaces, the clash of planes gives rise to body (three-dimensional).",
     "При переходе от плоскости к объему столкновение плоскостей «рождает» тело (трехмерное)."),
    ("In each case, the hierarchically important form or space is given meaning and significance by being an exception to the norm, an anomaly within an otherwise regular pattern..",
     "В любом случае иерархически важная форма или объем обретают смысл и  значительность, становясь исключением из нормы, «аномалией» в регулярной композиции."),
    ("Two points established in space by columnar elements or centralized forms can define an axis, an ordering device used throughout history to organize building forms and spaces.",
     "Между двумя точками, отмеченными в  пространстве столбообразными элементами или централизованными формами, устанавливается ось."),
    ("Stairs and certain modular materials, such as brick and concrete block, help us measure the scale of a space.",
     "Лестницы и некоторые модульные материалы вроде кирпичей или бетонных блоков облегчают оценку пространственного масштаба."),
    ("A special field that has developed from a concern with human factors is ergonomics—the applied science that coordinates the design of devices, systems, and environments with our physiological and psychological capacities and requirements.",
     "Внимание к  человеческим факторам породило новую прикладную науку — эргономику, которая занимается координацией дизайна разного рода устройств, систем и сред с нашими физиологическими и психологическими потребностями и возможностями."),
    ("From this module were derived the dimensions of the shaft, the capital, as well as the pedestal below and the entablature above, down to the smallest detail.",
     "Из  этого модуля выводились размеры ствола колонны, капители, базы и антаблемента  — все, вплоть до  мельчайших деталей."),
    ("The circle is a centralized, introverted figure that is normally stable and self-centering in its environment.",
     "Круг  — центричная, уравновешенная, замкнутая в себе фигура, образующая самостоятельный центр в пространстве."),
    ("A row of columns supporting an entablature—a colonnade—is often used to define the public face or facade of a building, especially one that fronts on a major civic space.",
     "Ряд колонн, несущих антаблемент, — колоннада, — нередко оформляет парадный фасад сооружения, особенно выходящий в  главное городское пространство."),
    ("In addition to the structural role columns play in supporting an overhead floor or roof plane, they can articulate the penetrable boundaries of spatial zones which mesh easily with adjacent spaces.",
     "Колонны не  только играют конструктивную роль опор, несущих верхний этаж или крышу, но и образуют сквозные, «решетчатые» границы пространственных зон, которые могут легко взаимодействовать с  окружающим пространством."),
    ("The term Building Information Modelling (BIM) is used to describe a collaborative process for the production and management of structured electronic information.",
     "Термин «информационное моделирование зданий» (Building Information Modelling — BIM) используется для описания совместного процесса производства и управления структурированной информацией в электронном формате."),
    ("Within the standard BIM approach, client requirements in terms of BIM services can take the form of an EIR or asset information requirement (AIR) document (as defined in PAS 1192-2:2013 and 1192-3:2014, respectively; BSI 2013b, 2014b).",
     "В рамках стандартного подхода к BIM, клиентские требования в части услуг по информационному моделированию могут предоставляться в форме электронного информационного ресурса или документа, содержащего требования по объекту информации (в терминах Спецификаций PAS 1192—2:2013 и 1192—3:2014 соответственно; Британского стандарта BSI 2013b, 2014b)."),
    ("A project information model (PIM) is the information model developed during the design and construction phase of a project, which often forms the basis of the AIM.",
     "Компьютерная модель проекта (КМП) — это модель, разработанная в ходе фазы проектирования и строительства проекта, часто составляющая основу компьютерной архитектурной модели."),
    ("The entrance facade operates at two scales: that of the building as a whole facing a public space and another at the size of a person entering the church.",
     "Фасадный вход воспринимается в двух масштабах: в масштабе городского сооружения, выходящего в общественное пространство, и в масштабе человека, входящего в церковь."),
    ("For architectural applications, computer modeling software enables the creation and manipulation of virtual, three-dimensional models of existing or proposed buildings and environments for analysis, testing, and appraisal.",
     "В архитектуре компьютерное моделирование используется для создания и  изменения виртуальных трехмерных моделей существующих или проектных зданий и  окружающего ландшафта с  целью анализа, тестирования и оценки."),
    ("The cantilever acknowledges the direction of approach along the longitudinal axis.",
     "Консоль указывает направление подхода вдоль продольных осей."),
    ("This dimensional transformation would create a hierarchical set of modules differentiated by size, proportion, and location.",
     "Подобные изменения в модульной соразмерности позволяют создавать иерархический ряд модулей, отличающихся по размерам, пропорциям и расположению."),
    ("Our perception of shape depends on the degree of visual contrast that exists along the contour separating a figure from its ground or between a form and its field.",
     "Наше восприятие очертаний зависит от  остроты визуального контраста, существующего между контурами фигуры и фоном."),
    ("The ground plane itself can be manipulated as well to establish a podium for a building form.",
     "Саму земную плоскость можно преобразовывать: превратить в подиум для святилища или иного значительного места."),
    ("Together, they form an inseparable reality — a unity of opposites—just as the elements of form and space together form the reality of architecture.",
     "Вместе они образуют нераздельную реальность — единство противоположностей, — точно так  же, как элементы формы и пространство вместе образуют архитектурную реальность."),
    ("Torii is a monumental, freestanding gateway on the approach to a Shinto shrine, consisting of two pillars connected at the top by a horizontal crosspiece and a lintel above it, usually curving upward.",
     "Тории представляют собой монументальные свободностоящие ворота на пути к синтоистскому святилищу, образованные двумя столбами, соединенными сверху деревянной поперечиной с карнизом над ней, обычно с загнутыми наверх краями."),
    ("Tokonoma, the spiritual center of a traditional Japanese house.",
     "Токонома — внутренний центр (фокус) в традиционном японском доме."),
    ("In sitations where greater visual and spatial continuity between two spaces is desired, even a change in level can establish a threshold and mark the passage from one place to another.",
     "В ситуациях, когда желательно большее визуальное и пространственное единство между двумя пространствами, достаточно перепада в уровнях, чтобы образовавшийся порог отметил переход из одного места в другое."),
    ("Vitruvius termed this a tetrastyle atrium.",
     "Витрувий назвал его «тетрастильным атриумом»."),
    ("An architectural composition can utilize symmetry to organize its forms and spaces in two ways.",
     "В архитектурной композиции симметрия может использоваться для организации форм и  пространств двумя способами."),
    ("The one-story space flows into the larger volume of which it is a part and to the outdoors.",
     "Пространство высотой в один этаж и  выходит на  улицу, и проникает в сам дом, становясь его частью."),
    ("A stele and tortoise guard the Tomb of Emperor Wan Li, northwest of Beijing, China.",
     "Стела и каменная черепаха охраняют гробницу императора Ван-Ли, к северо-западу от Пекина, Китай."),
    ("The pyramidal composition culminates in a spire that serves to establish this fortified monastery as a specific place in the landscape.",
     "Пирамидальная композиция завершается высоким шпилем, который указывает на исключительное место этого укрепленного монастыря в пейзаже."),
    ("Such elements as a table or chair, the risers and treads of a stairway, the sill of a window, and the lintel over a doorway, not only help us judge the size of a space but also give it a human scale.",
     "Такие объекты, как стол, стул, перила и ступени лестницы, подоконник, дверная притолока, не только помогают судить о размерах пространства, но и придают ему человеческий масштаб."),
    ("Believing that both humanity and the shrines housing their deities should belong to a higher universal order, they utilized these same proportions in their temple structures.",
     "Уверенные, что человеческие жилища и  святилища богов должны соответствовать высшему вселенскому порядку, они использовали те же пропорции в  храмовых сооружениях."),
    ("The form of the roof plane, in turn, is determined by the material, geometry, and proportions of its structural system and the manner in which it transfers its loads across space to its supports.",
     "В свою очередь, форма крыши определяется материалом, геометрией и  пропорциями общей структуры здания, а также способом передачи нагрузки на несущие элементы."),
    ("The enclosing planes may be ramparts, walls, or fences that isolate the field and exclude surrounding elements from the precinct.",
     "Ограждающими плоскостями могут быть валы, стены, изгороди или другие элементы, изолирующие пространственное поле от окружающей среды."),
    ("The form and scale of entrances and paths should also convey the functional and symbolic distinction between public promenades, private halls, and service corridors.",
     "Форма и масштаб входов и  проходов должны также отражать функциональные и символические различия между, скажем, общественным променадом, частным владением или служебным коридором."),
    ("The entrance portico of the library at the University of Virginia, modeled after the Pantheon in Rome, is scaled to the overall building form while the doorway and windows behind it are scaled to the size of the spaces within the building.",
     "Входной портик библиотеки университета Виргинии, построенный по образцу римского Пантеона, соответствует по масштабу общей форме здания, а расположенные за колоннами окна и двери отвечают масштабу внутренних помещений."),
    ("Plan of Peristyle Courtyard Houses on Delos, a Greek island in the Aegean.",
     "План перистильных домов с внутренними дворами на Делосе, греческом острове в Эгейском море."),
    ("Chinese painting illustrating the use of a pavilion structure to define a shaded resting place within an encampment.",
     "Этот китайский рисунок иллюстрирует, как используется павильонная система для организации затененной зоны отдыха в лагере."),
    ("The interior rooms surround a cortile— the principal courtyard of an Italian palazzo.",
     "Помещения расположены вокруг внутреннего дворика — главного элемента итальянского палаццо."),
    ("From the stupa, the Indian prototype, the pagoda gradually changed in form to resemble the traditional multistoried watch tower as it spread with Buddhism to China and Japan.",
     "Возникшая по  образцу индийской ступы, пагода постепенно изменяла форму, приближаясь к традиционной многоэтажной сторожевой башне, характерной для буддийских построек Китая и Японии."),
    ("To visibly mark a position in space or on the ground plane, a point must be projected vertically into a linear form, as a column, obelisk, or tower.",
     "Чтобы визуально отметить положение в  пространстве или на земной плоскости, точку надо продлить вертикально в виде линии — колонны, обелиска, башни."),
    ("Nuraghe at Palmavera, Sardinia, typical of the ancient stone towers of the Nuraghic culture, 18th–16th century B.C..",
     "Нураги, Пальмавера, Сардиния, типичные каменные башни и жилища древней культуры нурагов, XVIII—XVI вв. до н. э.."),
    ("U-shaped enclosures of space can range in scale from a niche in the wall of a room, to a hotel or dormitory room, and on to an arcaded outdoor space that organizes an entire complex of buildings.",
     "U-образные пространственные конфигурации могут использоваться в диапазоне от ниши в стене гостиной или спальни и до наружных аркад или галерей, организующих весь комплекс сооружений."),
    ("This view of the nave shows how rows of columns can provide a rhythmic measure of space.",
     "Здесь показано, как ряды колонн в корабле могут задавать ритм пространственного членения."),
    ("In the example illustrated to the left, four minaret towers outline a spatial field from which the dome of the Selim Mosque rises in splendor.",
     "Слева, например, мы можем увидеть четыре высоких минарета, которые зрительно выделяют участок пространства, где величественно высится купол мечети султана Селима."),
    ("In these examples, various forms of minarets mark the corners of a platform and establish a field of space—a three-dimensional framework—for the Mogul mausoleum structures.",
     "В этих примерах разнообразные формы минаретов отмечают углы подиума и  моделируют своеобразный трехмерный «каркас» пространственной зоны для зданий могольских мавзолеев."),
    ("The special effect of the hall in this firmly interlocked composition is produced by its great height, the barrelvaulted ceiling towering high above the side rooms into the mezzanine.",
     "Особый эффект этого зала, с его крепко увязанной композицией, создает его огромная высота, цилиндрический свод потолка, который возвышается над боковыми помещениями, превращаясь в мезони."),
    ("A prehistoric monument consisting of an upright megalith, usually standing alone but sometimes aligned with others.",
     "Доисторический монумент, представляет собой прямостоящий мегалит, обычно одиночный, но порой окруженный и другими камнями."),
    ("Shah Jahan built this white marble mausoleum for his favorite wife, Muntaz Mahal.",
     "Шах Джахан выстроил этот беломраморный мавзолей для своей любимой жены Мумтаз-Махал."),
    ("The form of solid mass rendered as a figure.",
     "Твердая масса оформляется как фигура."),
    ("This can be seen in the way joists are supported by beams, which in turn are supported by girders.",
     "Это отчетливо читается в том, как соотносятся несущие друг друга фермы, стропила и  балки."),
    ("The principle of hierarchy implies that in most if not all architectural compositions, real differences exist among their forms and spaces.",
     "Принцип иерархии подразумевает, что практически во  всех архитектурных композициях существуют реальные различия в  формах и  пространствах."),
    ("The flow of the space defined by parallel planes corresponds naturally to the paths of movement within a building, along its corridors, halls, and galleries.",
     "Направление пространственного потока между параллельными плоскостями, естественно, соответствует направлению движения внутри сооружения по коридорам, холлам, галереям."),
    ("To better comprehend the structure of a visual field, we tend to organize its elements into two opposing groups: positive elements, which are perceived as figures and negative elements, which provide a background for the figures.",
     "Чтобы лучше понять структуру зрительного поля, мы склонны делить эти элементы на две противоположные группы: позитивные элементы, которые воспринимаются в виде фигур, и негативные, которые составляют фон для фигур.")
]

TMX_PATH = "variant_14.tmx"


def build_tmx(pairs: Sequence[Tuple[str, str]], path: str,
              src_lang: str = "en", tgt_lang: str = "ru",
              domain: str = "Aerospace / UAS") -> str:
    """Сгенерировать валидную память переводов в формате TMX 1.4b.

    Args:
        pairs: последовательность кортежей ``(текст_источника, текст_перевода)``.
        path: путь к создаваемому файлу.
        src_lang: код языка оригинала (атрибут ``srclang`` заголовка).
        tgt_lang: код языка перевода.
        domain: значение свойства ``prop[type="domain"]``.

    Returns:
        Путь к созданному файлу.
    """
    created = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

    tmx = ET.Element("tmx", {"version": "1.4"})
    ET.SubElement(tmx, "header", {
        "creationtool": "lab01-generator",
        "creationtoolversion": "1.0",
        "segtype": "sentence",
        "o-tmf": "TMX",
        "adminlang": "en",
        "srclang": src_lang,
        "datatype": "plaintext",
        "creationdate": created,
    })
    body = ET.SubElement(tmx, "body")

    for index, (source, target) in enumerate(pairs, start=1):
        tu = ET.SubElement(body, "tu", {"tuid": str(index), "creationdate": created})
        ET.SubElement(tu, "prop", {"type": "domain"}).text = domain
        for lang, text in ((src_lang, source), (tgt_lang, target)):
            tuv = ET.SubElement(tu, "tuv", {XML_LANG: lang})
            ET.SubElement(tuv, "seg").text = text

    indent_tree(tmx)
    xml_body = ET.tostring(tmx, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write('<!DOCTYPE tmx SYSTEM "tmx14.dtd">\n')
        handler.write(xml_body)
    return path


build_tmx(PARALLEL_CORPUS, TMX_PATH)
print(f"Создан {TMX_PATH}: {os.path.getsize(TMX_PATH)} байт, единиц перевода: {len(PARALLEL_CORPUS)}\n")
print("--- первые 20 строк файла ---")
with open(TMX_PATH, encoding="utf-8") as handler:
    for line in list(handler)[:20]:
        print(line.rstrip())

Создан variant_14.tmx: 68642 байт, единиц перевода: 100

--- первые 20 строк файла ---
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE tmx SYSTEM "tmx14.dtd">
<tmx version="1.4">
  <header creationtool="lab01-generator" creationtoolversion="1.0" segtype="sentence" o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext" creationdate="20261019T003110Z"/>
  <body>
    <tu tuid="1" creationdate="20261019T003110Z">
      <prop type="domain">Aerospace / UAS</prop>
      <tuv xml:lang="en">
        <seg>The main characteristic of a BIM module is the representation of all components within a single structural unit, i.e. as a block having any geometric shape.</seg>
      </tuv>
      <tuv xml:lang="ru">
        <seg>Главная характеристика BIM-модуля — представление всех составляющих в одной структурной единице, т.е. в виде блока в любом геометрическом виде.</seg>
      </tuv>
    </tu>
    <tu tuid="2" creationdate="20261019T003110Z">
      <prop type="domain">Aerospace / UAS</prop>
     

## Блок 4. Задание — реализуйте четыре функции

Требования ко всем функциям: типизация из `typing`, соответствие PEP 8,
обработка отсутствующих элементов без падения (`None`-проверки).

**Подсказки:**

* язык секции читается как `element.get(XML_LANG)`, а не `element.get("lang")`;
* обход `<term>` внутри `langSet` удобно делать через `lang_set.iter("term")` —
  это покрывает и `tig`, и `ntig/termGrp`;
* текст сегмента лучше собирать через `"".join(seg.itertext())` — так не
  теряется текст при наличии инлайновых тегов;
* при подстановке терминов сортируйте их по убыванию длины.

In [48]:
def _parse_xml(path: str):
    """Безопасно разобрать XML-файл (без загрузки внешних DTD и сущностей)."""
    if LXML_AVAILABLE:
        parser = ET.XMLParser(resolve_entities=False, load_dtd=False, no_network=True)
        return ET.parse(path, parser)
    return ET.parse(path)


def validate_tbx(file_path: str) -> Dict[str, Any]:
    """Проверить структурную корректность TBX-файла.

    Проверяются: имя корневого элемента, наличие ``martifHeader``/``text``/``body``,
    наличие хотя бы двух ``langSet`` с непустым ``xml:lang`` в каждом ``termEntry``
    и наличие непустого ``<term>`` в каждой языковой секции.

    Args:
        file_path: путь к ``.tbx``-файлу.

    Returns:
        Словарь-отчёт с ключами ``valid`` (bool), ``errors`` (list[str]),
        ``entries`` (int), ``languages`` (list[str]).
    """
    errors: List[str] = []
    try:
        tree = _parse_xml(file_path)
    except Exception as exc:  # noqa: BLE001 - отчёт вместо аварийного завершения
        return {"valid": False, "errors": [f"XML не разобран: {exc}"],
                "entries": 0, "languages": []}

    root = tree.getroot()
    if root.tag != "martif":
        errors.append(f"Корневой элемент '{root.tag}', ожидается 'martif'")
    if root.find("martifHeader") is None:
        errors.append("Отсутствует секция <martifHeader>")

    body = root.find("./text/body")
    if body is None:
        errors.append("Отсутствует секция <text><body>")
        return {"valid": False, "errors": errors, "entries": 0, "languages": []}

    entries = body.findall("termEntry")
    if not entries:
        errors.append("В <body> нет ни одного <termEntry>")

    languages: List[str] = []
    for entry in entries:
        entry_id = entry.get("id", "<без id>")
        lang_sets = entry.findall("langSet")
        if len(lang_sets) < 2:
            errors.append(f"{entry_id}: языковых секций {len(lang_sets)}, нужно >= 2")
        for lang_set in lang_sets:
            lang = lang_set.get(XML_LANG)
            if not lang:
                errors.append(f"{entry_id}: у <langSet> нет атрибута xml:lang")
                continue
            if lang not in languages:
                languages.append(lang)
            terms = [t for t in lang_set.iter("term") if (t.text or "").strip()]
            if not terms:
                errors.append(f"{entry_id}/{lang}: нет непустого <term>")

    return {"valid": not errors, "errors": errors,
            "entries": len(entries), "languages": languages}


def parse_tbx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TBX-глоссарий в плоскую таблицу терминов.

    Args:
        file_path: путь к ``.tbx``-файлу.
        src_lang: код языка исходных терминов.
        tgt_lang: код языка переводных эквивалентов.

    Returns:
        ``pd.DataFrame`` с колонками ``entry_id``, ``source_term``, ``target_term``,
        ``pos``, ``definition``, ``subject_field``.
    """
    root = _parse_xml(file_path).getroot()
    rows: List[Dict[str, str]] = []

    for entry in root.iter("termEntry"):
        subject = ""
        descrip = entry.find("descrip[@type='subjectField']")
        if descrip is not None and descrip.text:
            subject = descrip.text.strip()

        by_lang: Dict[str, Dict[str, str]] = {}
        for lang_set in entry.findall("langSet"):
            lang = (lang_set.get(XML_LANG) or "").split("-")[0].lower()
            terms = [t.text.strip() for t in lang_set.iter("term") if (t.text or "").strip()]
            if not terms:
                continue
            note = lang_set.find(".//termNote[@type='partOfSpeech']")
            definition = lang_set.find(".//descrip[@type='definition']")
            by_lang[lang] = {
                "term": terms[0],
                "pos": (note.text or "").strip() if note is not None else "",
                "definition": (definition.text or "").strip() if definition is not None else "",
            }

        if src_lang in by_lang and tgt_lang in by_lang:
            rows.append({
                "entry_id": entry.get("id", ""),
                "source_term": by_lang[src_lang]["term"],
                "target_term": by_lang[tgt_lang]["term"],
                "pos": by_lang[src_lang]["pos"] or by_lang[tgt_lang]["pos"],
                "definition": by_lang[src_lang]["definition"],
                "subject_field": subject,
            })

    return pd.DataFrame(rows, columns=["entry_id", "source_term", "target_term",
                                       "pos", "definition", "subject_field"])
def _seg_text(tuv) -> str:
    """Достать текст сегмента, игнорируя инлайновые теги TMX (ph/bpt/ept/it)."""
    seg = tuv.find("seg")
    if seg is None:
        return ""
    return re.sub(r"\s+", " ", "".join(seg.itertext())).strip()


def parse_tmx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TMX-память переводов в таблицу параллельных сегментов.

    Args:
        file_path: путь к ``.tmx``-файлу.
        src_lang: код исходного языка.
        tgt_lang: код языка перевода.

    Returns:
        ``pd.DataFrame`` с колонками ``tuid``, ``source_segment``, ``target_segment``,
        ``domain``, ``src_len``. Строки без одного из языков отбрасываются.
    """
    root = _parse_xml(file_path).getroot()
    rows: List[Dict[str, Any]] = []

    for order, tu in enumerate(root.iter("tu"), start=1):
        segments: Dict[str, str] = {}
        for tuv in tu.findall("tuv"):
            lang = (tuv.get(XML_LANG) or tuv.get("lang") or "").split("-")[0].lower()
            text = _seg_text(tuv)
            if lang and text:
                segments[lang] = text

        if src_lang not in segments or tgt_lang not in segments:
            continue

        domain_node = tu.find("prop[@type='domain']")
        rows.append({
            "tuid": tu.get("tuid", str(order)),
            "source_segment": segments[src_lang],
            "target_segment": segments[tgt_lang],
            "domain": (domain_node.text or "").strip() if domain_node is not None else "",
            "src_len": len(segments[src_lang]),
        })

    frame = pd.DataFrame(rows, columns=["tuid", "source_segment", "target_segment",
                                        "domain", "src_len"])
    # предыдущий сегмент нужен для контекстного сопоставления (ICE)
    frame["prev_source"] = frame["source_segment"].shift(1).fillna("")
    return frame

def similarity(a: str, b: str) -> Dict[str, float]:
    """Вернуть словарь метрик сходства: ``lev``, ``token_sort``, ``score``."""
    norm_a, norm_b = normalize(a), normalize(b)
    lev = levenshtein_ratio(norm_a, norm_b)
    tsr = token_sort_ratio(a, b)
    return {"lev": lev, "token_sort": tsr, "score": max(lev, tsr)}


def classify_match(score: float, context_matched: bool) -> str:
    """Определить тип совпадения по проценту и признаку совпадения контекста."""
    if score >= 100.0:
        return "ICE (101%)" if context_matched else "Exact (100%)"
    if score >= 95.0:
        return "High Fuzzy"
    if score >= 75.0:
        return "Fuzzy"
    return "Low Fuzzy"


def calculate_fuzzy_match(source_query: str, tm_df: pd.DataFrame,
                          threshold: float = 70.0,
                          context_before: Optional[str] = None,
                          top_n: int = 5) -> List[Dict[str, Any]]:
    """Найти в памяти переводов сегменты, похожие на запрос.

    Args:
        source_query: сегмент, который нужно перевести.
        tm_df: таблица памяти переводов (результат :func:`parse_tmx`).
        threshold: минимальный процент сходства для попадания в выдачу.
        context_before: предыдущий сегмент документа; при совпадении с
            ``prev_source`` 100-процентное совпадение повышается до ICE.
        top_n: максимальное число возвращаемых кандидатов.

    Returns:
        Список словарей, отсортированный по убыванию ``score``; ключи:
        ``tuid``, ``source_segment``, ``target_segment``, ``score``,
        ``lev_ratio``, ``token_sort_ratio``, ``match_type``.
    """
    if tm_df.empty:
        return []

    results: List[Dict[str, Any]] = []
    for _, row in tm_df.iterrows():
        metrics = similarity(source_query, row["source_segment"])
        if metrics["score"] < threshold:
            continue
        context_matched = bool(
            context_before is not None
            and normalize(context_before) == normalize(str(row.get("prev_source", "")))
        )
        results.append({
            "tuid": row["tuid"],
            "source_segment": row["source_segment"],
            "target_segment": row["target_segment"],
            "score": metrics["score"],
            "lev_ratio": metrics["lev"],
            "token_sort_ratio": metrics["token_sort"],
            "match_type": classify_match(metrics["score"], context_matched),
        })

    results.sort(key=lambda item: (item["score"], item["tuid"]), reverse=True)
    return results[:top_n]

def apply_term_glossary(text: str, glossary_df: pd.DataFrame,
                        template: str = "{source} [{target}]") -> str:
    """Разметить исходный текст переводными эквивалентами из глоссария.

    Args:
        text: исходный сегмент на языке оригинала.
        glossary_df: таблица глоссария (результат :func:`parse_tbx`) с колонками
            ``source_term`` и ``target_term``.
        template: шаблон подстановки; поля ``{source}`` и ``{target}``.

    Returns:
        Текст, в котором найденные термины заменены по шаблону. Порядок слов и
        регистр исходного текста сохраняются.
    """
    if glossary_df.empty or not text:
        return text

    pairs = sorted(
        zip(glossary_df["source_term"], glossary_df["target_term"]),
        key=lambda pair: len(pair[0]), reverse=True,
    )

    protected: List[str] = []

    def _store(replacement: str) -> str:
        protected.append(replacement)
        return f"\x00{len(protected) - 1}\x00"

    result = text
    for source_term, target_term in pairs:
        pattern = re.compile(rf"(?<!\w){re.escape(source_term)}(?!\w)", re.IGNORECASE)
        result = pattern.sub(
            lambda match: _store(template.format(source=match.group(0), target=target_term)),
            result,
        )

    return re.sub(r"\x00(\d+)\x00", lambda m: protected[int(m.group(1))], result)



### 4.1. Вспомогательные функции (реализуйте при необходимости)

Эти функции не оцениваются отдельно, но нужны для `calculate_fuzzy_match`.

In [49]:
_PUNCT_RE = re.compile(r"[^\w\s]", re.UNICODE)

def normalize(text: str) -> str:
    """Привести строку к нормальной форме: NFKC, нижний регистр, без пунктуации."""
    text = unicodedata.normalize("NFKC", text).lower().replace("ё", "е")
    text = _PUNCT_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def levenshtein_ratio(a: str, b: str) -> float:
    """Процент сходства двух строк на основе расстояния Левенштейна (0–100)."""
    if not a and not b:
        return 100.0
    longest = max(len(a), len(b))
    return round((1.0 - edit_distance(a, b) / longest) * 100.0, 2)


def token_sort_ratio(a: str, b: str) -> float:
    """Процент сходства при игнорировании порядка слов (0–100)."""
    sorted_a = " ".join(sorted(normalize(a).split()))
    sorted_b = " ".join(sorted(normalize(b).split()))
    return levenshtein_ratio(sorted_a, sorted_b)

## Блок 5. Автотесты

Запустите ячейки ниже **после** реализации функций. Все проверки должны
завершиться без `AssertionError`. Тесты работают на демо-файлах из Блока 3.

In [50]:
# --- Тест 1: метрики сходства -------------------------------------------
assert abs(levenshtein_ratio("telemetry", "telemetry") - 100.0) < 1e-6, \
    "Одинаковые строки должны давать 100 %"
assert levenshtein_ratio("", "") == 100.0, "Две пустые строки — 100 %"
assert 0.0 <= levenshtein_ratio("payload", "airframe") < 50.0, \
    "Непохожие строки должны давать низкий процент"
assert abs(token_sort_ratio("angle of attack", "attack of angle") - 100.0) < 1e-6, \
    "Token Sort Ratio должен игнорировать порядок слов"
assert normalize("  Angle   OF ATTACK, critical! ") == "angle of attack critical", \
    "normalize: нижний регистр, без пунктуации, одиночные пробелы"
print("Тест 1 (метрики сходства) — OK")

Тест 1 (метрики сходства) — OK


In [51]:
# --- Тест 2: парсинг TBX -------------------------------------------------
glossary_df = parse_tbx("demo.tbx")

assert isinstance(glossary_df, pd.DataFrame), "parse_tbx должен вернуть DataFrame"
assert len(glossary_df) == 3, f"Ожидается 3 термина, получено {len(glossary_df)}"
for column in ["entry_id", "source_term", "target_term", "pos",
               "definition", "subject_field"]:
    assert column in glossary_df.columns, f"Нет колонки '{column}'"

row = glossary_df.loc[glossary_df["source_term"] == "flight controller"].iloc[0]
assert row["target_term"] == "полётный контроллер", "Неверный русский эквивалент"
assert row["entry_id"] == "tid-001", "Не извлечён атрибут id у termEntry"
assert row["pos"] == "noun", "Не извлечён termNote[@type='partOfSpeech']"
assert "telemetry link" in set(glossary_df["source_term"]), \
    "Термин из <tig> без termNote должен быть извлечён"
print("Тест 2 (парсинг TBX) — OK")
glossary_df.head()

Тест 2 (парсинг TBX) — OK


,entry_id,source_term,target_term,pos,definition,subject_field
0,tid-001,flight controller,полётный контроллер,noun,Onboard flight computer.,Demo
1,tid-002,payload,целевая нагрузка,noun,,Demo
2,tid-003,telemetry link,канал телеметрии,,,Demo


In [52]:
# --- Тест 3: парсинг TMX -------------------------------------------------
tm_df = parse_tmx("demo.tmx")

assert isinstance(tm_df, pd.DataFrame), "parse_tmx должен вернуть DataFrame"
assert len(tm_df) == 3, f"Ожидается 3 единицы перевода, получено {len(tm_df)}"
for column in ["tuid", "source_segment", "target_segment", "domain",
               "src_len", "prev_source"]:
    assert column in tm_df.columns, f"Нет колонки '{column}'"

first = tm_df.iloc[0]
assert first["source_segment"].startswith("The flight controller"), \
    "Не извлечён текст <seg> исходного языка"
assert first["target_segment"].startswith("Полётный контроллер"), \
    "Не извлечён текст <seg> языка перевода"
assert first["prev_source"] == "", "У первой строки prev_source должен быть пустым"
assert tm_df.iloc[1]["prev_source"] == first["source_segment"], \
    "prev_source должен содержать предыдущий исходный сегмент"
assert first["src_len"] == len(first["source_segment"]), "Неверная длина сегмента"
print("Тест 3 (парсинг TMX) — OK")
tm_df.head()

Тест 3 (парсинг TMX) — OK


,tuid,source_segment,target_segment,domain,src_len,prev_source
0,1,The flight controller stabilises the aircr...,Полётный контроллер стабилизирует аппарат.,Demo,46,
1,2,The operator checks the telemetry link.,Оператор проверяет канал телеметрии.,Demo,39,The flight controller stabilises the aircr...
2,3,The payload is mounted under the fuselage.,Целевая нагрузка устанавливается под фюзел...,Demo,42,The operator checks the telemetry link.


In [53]:
# --- Тест 4: Fuzzy Match и ICE ------------------------------------------
exact_query = "The operator checks the telemetry link."
matches = calculate_fuzzy_match(exact_query, tm_df, threshold=70.0)
assert matches, "Точное совпадение обязано быть найдено"
assert abs(matches[0]["score"] - 100.0) < 1e-6, "Точное совпадение — 100 %"
assert matches[0]["match_type"] == "Exact (100%)", \
    "Без контекста 100 % — это Exact, а не ICE"

ice = calculate_fuzzy_match(
    exact_query, tm_df, threshold=70.0,
    context_before="The flight controller stabilises the aircraft.")
assert ice[0]["match_type"] == "ICE (101%)", \
    "При совпадении предыдущего сегмента 100 % должно стать ICE"

fuzzy = calculate_fuzzy_match("The engineer checks the telemetry link.",
                              tm_df, threshold=70.0)
assert fuzzy, "Похожий сегмент должен попадать в выдачу"
assert 75.0 <= fuzzy[0]["score"] < 100.0, \
    f"Ожидается Fuzzy 75–99 %, получено {fuzzy[0]['score']}"
assert fuzzy[0]["match_type"] in {"Fuzzy", "High Fuzzy"}, "Неверный тип совпадения"

scores = [m["score"] for m in calculate_fuzzy_match("telemetry link check",
                                                    tm_df, threshold=0.0)]
assert scores == sorted(scores, reverse=True), \
    "Результаты должны быть отсортированы по убыванию сходства"
assert calculate_fuzzy_match("Совершенно посторонний текст про бухгалтерию.",
                             tm_df, threshold=75.0) == [], \
    "Ниже порога результатов быть не должно"
assert len(calculate_fuzzy_match("telemetry", tm_df, threshold=0.0, top_n=2)) <= 2, \
    "Параметр top_n не учитывается"
print("Тест 4 (Fuzzy Match и ICE) — OK")

Тест 4 (Fuzzy Match и ICE) — OK


In [54]:
# --- Тест 5: подстановка терминов ---------------------------------------
marked = apply_term_glossary("The flight controller reads the payload status.",
                             glossary_df)
assert "[полётный контроллер]" in marked, "Термин из глоссария не подставлен"
assert "[целевая нагрузка]" in marked, "Подставлен не каждый найденный термин"
assert "flight controller" in marked, "Исходный термин должен сохраняться"

assert apply_term_glossary("No terms here at all.", glossary_df) == \
    "No terms here at all.", "Текст без терминов не должен меняться"

case_test = apply_term_glossary("Payload mass is 2 kg.", glossary_df)
assert "Payload [целевая нагрузка]" in case_test, \
    "Поиск должен быть регистронезависимым, а регистр текста — сохраняться"

assert "[целевая нагрузка]" not in apply_term_glossary("payloads are heavy",
                                                       glossary_df), \
    "Совпадение должно быть по границам слов, а не по подстроке"
print("Тест 5 (подстановка терминов) — OK")
print(marked)

Тест 5 (подстановка терминов) — OK
The flight controller [полётный контроллер] reads the payload [целевая нагрузка] status.


## Блок 6. Работа со своим вариантом

После прохождения всех тестов выполните задание на данных своего варианта.

In [58]:
VARIANT = 14
SURNAME = "Shishkova"

TBX_FILE = f"variant_{VARIANT}.tbx"   # не менее 15 терминов
TMX_FILE = f"variant_{VARIANT}.tmx"   # не менее 10 параллельных предложений

my_glossary = parse_tbx(TBX_FILE)
my_tm = parse_tmx(TMX_FILE)

QUERIES: List[Dict[str, Optional[str]]] = [
    {
        "label": "1. ICE — точное совпадение с совпадением контекста",
        "query": "A stele and tortoise guard the Tomb of Emperor Wan Li, northwest of Beijing, China.",
        "context": "The one-story space flows into the larger volume of which it is a part and to the outdoors.",
    },
    {
        "label": "2. Fuzzy — заменены два слова",
        "query": "The intent of all theories of parts is to create a sense of order and harmony among the symbols in a visual construction.",
        "context": None,
    },
    {
        "label": "3. No match — сегмент вне предметной области",
        "query": "The quarterly financial report was approved by the board.",
        "context": None,
    },
]

all_matches: List[Dict[str, Any]] = []
for case in QUERIES:
    matches = calculate_fuzzy_match(case["query"], my_tm, threshold=70.0,
                                    context_before=case["context"])
    print("=" * 100)
    print(case["label"])
    print(f"Запрос: {case['query']}")
    if not matches:
        print("  -> совпадений выше порога 70 % не найдено, сегмент переводится с нуля")
        continue
    for rank, match in enumerate(matches, start=1):
        print(f"  #{rank} {match['score']:6.2f} %  [{match['match_type']:<12}] "
              f"tuid={match['tuid']}")
        print(f"      TM src: {match['source_segment']}")
        print(f"      TM tgt: {match['target_segment']}")
        match["query"] = case["query"]
        all_matches.append(match)

matches_df = pd.DataFrame(all_matches)

GLOSSARY_CSV = "glossary_smartcat_v14.csv"
TM_CSV = "tm_segments_v14.csv"
FUZZY_CSV = "fuzzy_report_v14.csv"

smartcat_df = pd.DataFrame({
    "Source (en)": my_glossary["source_term"],
    "Target (ru)": my_glossary["target_term"],
    "Comment": my_glossary["definition"],
})
smartcat_df.to_csv(GLOSSARY_CSV, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)

my_tm[["tuid", "source_segment", "target_segment", "domain"]].to_csv(TM_CSV, index=False, encoding="utf-8-sig")

matches_df[["query", "tuid", "source_segment", "target_segment",
            "score", "lev_ratio", "token_sort_ratio", "match_type"]].to_csv(FUZZY_CSV, index=False, encoding="utf-8-sig")

print("\nЭкспортированные файлы:")
for name in [TBX_PATH, TMX_PATH, GLOSSARY_CSV, TM_CSV, FUZZY_CSV]:
    print(f"  {name:<28} {os.path.getsize(name):>7} байт")

print("=" * 100)
print(f"\nВсего кандидатов выше порога: {len(matches_df)}")
matches_df[["query", "tuid", "score", "lev_ratio", "token_sort_ratio", "match_type"]].head(6)

1. ICE — точное совпадение с совпадением контекста
Запрос: A stele and tortoise guard the Tomb of Emperor Wan Li, northwest of Beijing, China.
  #1 100.00 %  [ICE (101%)  ] tuid=75
      TM src: A stele and tortoise guard the Tomb of Emperor Wan Li, northwest of Beijing, China.
      TM tgt: Стела и каменная черепаха охраняют гробницу императора Ван-Ли, к северо-западу от Пекина, Китай.
2. Fuzzy — заменены два слова
Запрос: The intent of all theories of parts is to create a sense of order and harmony among the symbols in a visual construction.
  #1  89.76 %  [Fuzzy       ] tuid=36
      TM src: The intent of all theories of proportions is to create a sense of order and harmony among the elements in a visual construction.
      TM tgt: Цель всех пропорциональных теорий - создать впечатление упорядоченности и гармонии всех зримых элементов постройки.
3. No match — сегмент вне предметной области
Запрос: The quarterly financial report was approved by the board.
  -> совпадений выше порога 

,query,tuid,score,lev_ratio,token_sort_ratio,match_type
0,A stele and tortoise guard the Tomb of Emp...,75,100.00,100.00,100.0,ICE (101%)
1,The intent of all theories of parts is to ...,36,89.76,89.76,81.1,Fuzzy


## Блок 7. Выводы

Ответьте письменно (3–5 предложений на пункт):

1. Чем концептуально-ориентированная организация TBX отличается от
   сегментно-ориентированной организации TMX?
2. Почему `xml:lang` требует особой обработки при парсинге и как вы её решили?
3. В каких случаях Levenshtein ratio даёт заниженную оценку сходства и как это
   компенсирует Token Sort Ratio?
4. Чем ICE (101 %) отличается от обычного 100 %-совпадения и почему это важно
   для стоимости проекта в Smartcat?
5. Какие проблемы возникли при автоматической подстановке терминов и как вы их
   решили?

**Ваши выводы:**

_______________________________________________

---

### Что сдать на проверку

1. Ссылка на Colab-ноутбук `lab_01_Фамилия.ipynb` (доступ «для всех по ссылке»).
2. Файлы `variant_NN.tbx`, `variant_NN.tmx`, `glossary_smartcat.csv`.
3. Отчёт с выводами и скриншотом импортированного глоссария в Smartcat.